In [1]:
!pip install seqeval evaluate -q

In [2]:
import json
import pandas as pd
import numpy as np
import argparse
from itertools import chain
from functools import partial

import torch
from transformers import AutoTokenizer, Trainer, TrainingArguments
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification
import evaluate
from datasets import Dataset, features

from seqeval.metrics import recall_score, precision_score
from seqeval.metrics import classification_report
from seqeval.metrics import f1_score

In [3]:
train_data = json.load(open('/content/train.json'))

In [4]:
all_labels = sorted(list(set(chain(*[x["labels"] for x in train_data]))))
label2id = {l: i for i,l in enumerate(all_labels)}
id2label = {v:k for k,v in label2id.items()}

target = [item for item in all_labels if item != 'O']

print(id2label)

{0: 'B-EMAIL', 1: 'B-ID_NUM', 2: 'B-NAME_STUDENT', 3: 'B-PHONE_NUM', 4: 'B-STREET_ADDRESS', 5: 'B-URL_PERSONAL', 6: 'B-USERNAME', 7: 'I-ID_NUM', 8: 'I-NAME_STUDENT', 9: 'I-PHONE_NUM', 10: 'I-STREET_ADDRESS', 11: 'I-URL_PERSONAL', 12: 'O'}


In [5]:
# Labelize each character of each token to rebuild indexes after model's tokenization
def rebuild_text(data):

    text, labels = [], []

    for tok, lab, ws in zip(
        data["tokens"], data["provided_labels"], data["trailing_whitespace"]
    ):
        # append each token to the reconstructed text and the label for each token's character
        text.append(tok)
        labels.extend([lab] * len(tok))

        # add space in text if whitespace and label "O"
        if ws:
            text.append(" ")
            labels.append("O")

    return text, labels



In [6]:
# Prepare data to be fed to the model & attribute labels to new token format
def tokenize(data, tokenizer, label2id, max_length):

    text, labels = rebuild_text(data)
    text = "".join(text)
    labels = np.array(labels)
    token_labels = []

    # returns a dictionary-like object containing tokenized inputs and offsets mapping (represents the mapping between the tokens and their corresponding positions in the original text)
    tokenized = tokenizer(text, return_offsets_mapping=True, max_length=max_length)

    for start_idx, end_idx in tokenized.offset_mapping:

        # if CLS tokens
        if start_idx == 0 and end_idx == 0:
            token_labels.append(label2id["O"])
            continue

        # if token starts with ws
        if text[start_idx].isspace():
            start_idx += 1

        token_labels.append(label2id[labels[start_idx]])

    length = len(tokenized.input_ids)

    return {**tokenized, "labels": token_labels, "length": length}

In [7]:
TRAINING_MODEL_PATH = "microsoft/deberta-v3-base"
TRAINING_MAX_LENGTH = 1024
OUTPUT_DIR = "output"

In [8]:
tokenizer = AutoTokenizer.from_pretrained(TRAINING_MODEL_PATH)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

In [9]:
ds = Dataset.from_dict({
    "full_text": [x["full_text"] for x in train_data],
    "document": [str(x["document"]) for x in train_data],
    "tokens": [x["tokens"] for x in train_data],
    "trailing_whitespace": [x["trailing_whitespace"] for x in train_data],
    "provided_labels": [x["labels"] for x in train_data],
})

In [11]:
ds = ds.map(tokenize, fn_kwargs={"tokenizer":tokenizer, "label2id":label2id, "max_length":TRAINING_MAX_LENGTH}, num_proc=3)

Map (num_proc=3):   0%|          | 0/6807 [00:00<?, ? examples/s]

In [12]:
# Compare tokens and labels for original dataset and new tokenization
x = ds[0]

for t,l in zip(x["tokens"], x["provided_labels"]):
    if l != "O":
        print((t,l))

print("*"*100)

for t, l in zip(tokenizer.convert_ids_to_tokens(x["input_ids"]), x["labels"]):
    if id2label[l] != "O":
        print((t,id2label[l]))

('Nathalie', 'B-NAME_STUDENT')
('Sylla', 'I-NAME_STUDENT')
('Nathalie', 'B-NAME_STUDENT')
('Sylla', 'I-NAME_STUDENT')
('Nathalie', 'B-NAME_STUDENT')
('Sylla', 'I-NAME_STUDENT')
****************************************************************************************************
('N', 'B-NAME_STUDENT')
('atha', 'B-NAME_STUDENT')
('lie', 'B-NAME_STUDENT')
('▁S', 'I-NAME_STUDENT')
('ylla', 'I-NAME_STUDENT')
('N', 'B-NAME_STUDENT')
('atha', 'B-NAME_STUDENT')
('lie', 'B-NAME_STUDENT')
('▁S', 'I-NAME_STUDENT')
('ylla', 'I-NAME_STUDENT')
('N', 'B-NAME_STUDENT')
('atha', 'B-NAME_STUDENT')
('lie', 'B-NAME_STUDENT')
('▁S', 'I-NAME_STUDENT')
('ylla', 'I-NAME_STUDENT')


In [18]:
#Metrics

def compute_metrics(p, all_labels):
    # p is a tuple containing preds and true labels
    predictions, labels = p
    # preds are in form of probs for each label for each token => we take the highest one
    predictions = np.argmax(predictions, axis=2)

    # Remove special tokens from preds and labels
    true_predictions = [
        [all_labels[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [all_labels[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Compute metrics using sklearn and own formula
    recall = recall_score(true_labels, true_predictions)
    precision = precision_score(true_labels, true_predictions)
    f1_score = (1 + 5*5) * recall * precision / (5*5*precision + recall)

    # Store metrics and return
    results = {
        'recall': recall,
        'precision': precision,
        'f1': f1_score
    }

    return results


In [13]:
# Model
model = AutoModelForTokenClassification.from_pretrained(
    TRAINING_MODEL_PATH,
    num_labels=len(all_labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task

In [14]:
# Creates a collator object (tailored for token classification tasks)
collator = DataCollatorForTokenClassification(tokenizer, pad_to_multiple_of=16)

In [42]:
import math

# Define training arguments
total_steps = (len(ds) // (4 * 2)) * 1 # (dataset_size // (per_device_train_batch_size * gradient_accumulation_steps)) * num_train_epochs

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    fp16=False,
    learning_rate=2e-5,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    report_to="none",
    eval_strategy="no",
    do_eval=False,
    save_total_limit=1,
    logging_steps=20,
    lr_scheduler_type='cosine',
    metric_for_best_model="f1",
    greater_is_better=True,
    warmup_steps=math.ceil(total_steps * 0.1), # calculate warmup_steps from warmup_ratio=0.1
    weight_decay=0.01
)

In [43]:
# Define trainer object (responsible for orchestrating the training process)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds,
    data_collator=collator,
    processing_class=tokenizer,
    compute_metrics=partial(compute_metrics, all_labels=all_labels),
)


In [44]:
%%time
trainer.train()

Step,Training Loss
20,0.000000
40,0.000000
60,0.000000
80,0.000000
100,0.000000
120,0.000000
140,0.000000
160,0.000000
180,0.000000
200,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CPU times: user 13min 59s, sys: 2min 53s, total: 16min 53s
Wall time: 17min 36s


TrainOutput(global_step=851, training_loss=0.0, metrics={'train_runtime': 1055.8627, 'train_samples_per_second': 6.447, 'train_steps_per_second': 0.806, 'total_flos': 3167966649980064.0, 'train_loss': 0.0, 'epoch': 1.0})

In [45]:
trainer.save_model("deberta3base_1024")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [46]:
tokenizer.save_pretrained("deberta3base_1024")

('deberta3base_1024/tokenizer_config.json', 'deberta3base_1024/tokenizer.json')

### Make Predictions on Test Data

In [47]:
# Load test data
test_data = json.load(open('/content/test.json'))

In [48]:
# Define a tokenization function for inference
# This function reconstructs the text from tokens and trailing whitespaces
# and also stores the character spans of original tokens for later alignment.
def tokenize_for_inference(example, tokenizer, max_length):
    text_parts = []
    original_token_spans = [] # Store char spans for original tokens in the reconstructed text
    current_char_offset = 0
    for tok, ws in zip(example["tokens"], example["trailing_whitespace"]):
        text_parts.append(tok)
        original_token_spans.append((current_char_offset, current_char_offset + len(tok)))
        current_char_offset += len(tok)
        if ws:
            text_parts.append(" ")
            current_char_offset += 1
    reconstructed_text = "".join(text_parts)

    # Tokenize the reconstructed text
    tokenized = tokenizer(reconstructed_text, return_offsets_mapping=True, truncation=True, max_length=max_length)
    tokenized["length"] = len(tokenized.input_ids)
    # Store the reconstructed text and original token spans in the tokenized output for easy reference
    tokenized["reconstructed_text"] = reconstructed_text
    tokenized["original_token_spans"] = original_token_spans
    return tokenized

# Create a Dataset for test data
test_ds = Dataset.from_dict({
    "full_text": [x["full_text"] for x in test_data],
    "document": [str(x["document"]) for x in test_data],
    "tokens": [x["tokens"] for x in test_data],
    "trailing_whitespace": [x["trailing_whitespace"] for x in test_data],
})

# Apply the tokenization function to the test dataset
test_ds = test_ds.map(tokenize_for_inference, fn_kwargs={
    "tokenizer": tokenizer,
    "max_length": TRAINING_MAX_LENGTH
}, num_proc=3)

Map (num_proc=3):   0%|          | 0/10 [00:00<?, ? examples/s]

In [49]:
%%time
# Make predictions on the tokenized test dataset
predictions_output = trainer.predict(test_ds)

# Extract predicted logits and convert to label IDs
all_pred_logits = predictions_output.predictions
all_pred_ids = np.argmax(all_pred_logits, axis=2)

CPU times: user 713 ms, sys: 3.35 ms, total: 716 ms
Wall time: 730 ms


In [50]:
# Post-process predictions to align with original tokens and generate submission.csv
results = []

for doc_idx, example in enumerate(test_data):
    document_id = example["document"]
    original_tokens = example["tokens"]

    tokenized_example = test_ds[doc_idx] # Get the tokenized data for this document
    offset_mapping = tokenized_example["offset_mapping"]
    original_token_spans = tokenized_example["original_token_spans"]
    pred_ids_for_doc = all_pred_ids[doc_idx] # Get the predicted label IDs for this document

    # Initialize labels for all original tokens as 'O'
    original_token_labels = ['O'] * len(original_tokens)

    # Iterate through each original token's character span
    for original_tok_idx, (orig_char_start, orig_char_end) in enumerate(original_token_spans):
        sub_token_preds_for_this_orig_token = []

        # Iterate through the tokenized sub-tokens and their predictions
        for seq_idx, (sub_char_start, sub_char_end) in enumerate(offset_mapping):
            # Skip special tokens (where start and end offsets are the same)
            if sub_char_start == sub_char_end:
                continue

            # Check for overlap between the sub-token's span and the original token's span
            # If there's any overlap, consider this sub-token's prediction for the original token
            if max(orig_char_start, sub_char_start) < min(orig_char_end, sub_char_end):
                sub_token_preds_for_this_orig_token.append(id2label[pred_ids_for_doc[seq_idx]])

        # Determine the final label for the original token
        # Simple heuristic: take the first non-'O' label found among its sub-tokens.
        # This generally works well for BIO schemes where B- takes precedence.
        final_label = 'O'
        for pred_lbl in sub_token_preds_for_this_orig_token:
            if pred_lbl != 'O':
                final_label = pred_lbl # Assign the label and break
                break

        original_token_labels[original_tok_idx] = final_label

    # Collect results in the submission format (document, token, label)
    for token_idx, label in enumerate(original_token_labels):
        if label != 'O': # Only include non-'O' predictions in the submission
            results.append({
                "document": document_id,
                "token": token_idx,
                "label": label
            })

# Create a pandas DataFrame from the results and add 'row_id' using reset_index()
submission_df = pd.DataFrame(results).reset_index().rename(columns={'index': 'row_id'})

# Reorder columns to match the desired format: row_id, document, token, label
submission_df = submission_df[['row_id', 'document', 'token', 'label']]

# Save to submission.csv
submission_df.to_csv("submission.csv", index=False)

print("Submission file 'submission.csv' generated successfully.")

# Display the first few rows of the submission file
display(submission_df.head())

Submission file 'submission.csv' generated successfully.


,document,token,label
0,7,0,B-EMAIL
1,7,1,B-EMAIL
2,7,2,B-EMAIL
3,7,3,B-EMAIL
4,7,4,B-EMAIL


In [51]:
len(test_data)

10